In [1]:
import json
import numpy as np
import random

import psalm_scraper as ps
import psalm_utils as pu

from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
ps_dict = ps.load_ps_dict('ps_verses.json')

In [3]:
total_verses = sum(len(ps_dict[p]) for p in ps_dict)
positive = sum(1 for p in ps_dict for v in ps_dict[p] if v['label'] == 1)
negative = sum(1 for p in ps_dict for v in ps_dict[p] if v['label'] == 0)
neutral = sum(1 for p in ps_dict for v in ps_dict[p] if v['label'] is None)

print(f"Total verses: {total_verses}")
print(f"Positive (1): {positive} ({positive/total_verses:.2%})")
print(f"Negative (0): {negative} ({negative/total_verses:.2%})")
print(f"Neutral (None): {neutral} ({neutral/total_verses:.2%})")
print(f"Non-neutral for training: {positive + negative} ({(positive + negative)/total_verses:.2%})")

Total verses: 2457
Positive (1): 817 (33.25%)
Negative (0): 355 (14.45%)
Neutral (None): 1285 (52.30%)
Non-neutral for training: 1172 (47.70%)


In [4]:
pu.add_binary_labels(ps_dict, pos_threshold=0.5, neg_threshold=-0.5)
ps.save_ps_dict(ps_dict, 'ps_verses.json')

total_verses = sum(len(ps_dict[p]) for p in ps_dict)
positive = sum(1 for p in ps_dict for v in ps_dict[p] if v['label'] == 1)
negative = sum(1 for p in ps_dict for v in ps_dict[p] if v['label'] == 0)
neutral = sum(1 for p in ps_dict for v in ps_dict[p] if v['label'] is None)

print(f"Total verses: {total_verses}")
print(f"Positive (1): {positive} ({positive/total_verses:.2%})")
print(f"Negative (0): {negative} ({negative/total_verses:.2%})")
print(f"Neutral (None): {neutral} ({neutral/total_verses:.2%})")
print(f"Non-neutral for training: {positive + negative} ({(positive + negative)/total_verses:.2%})")

Total verses: 2457
Positive (1): 817 (33.25%)
Negative (0): 355 (14.45%)
Neutral (None): 1285 (52.30%)
Non-neutral for training: 1172 (47.70%)


In [5]:
training_data = [(v['text'], v['label']) for p in ps_dict for v in ps_dict[p] if v['label'] is not None]
texts, labels = zip(*training_data)
psalms = np.array(texts)
outcomes = np.array(labels)

# Train/test split
random.seed(42)
indices = list(range(len(psalms)))
sample_size = int(len(indices) * 0.8)
training_idx = random.sample(indices, k=sample_size)
test_idx = [i for i in indices if i not in training_idx]
ps_train, y_train = psalms[training_idx], outcomes[training_idx]
ps_test, y_test = psalms[test_idx], outcomes[test_idx]

freqs = pu.build_freqs(ps_train, y_train)
d_pos = np.sum(y_train)
d_neg = np.sum(1 - y_train)
log_prior = np.log(d_pos / d_neg) if d_neg != 0 else float('inf')
vocab = [x[0] for x in set(freqs.keys())]
V = len(set(vocab))
N_pos, N_neg = 0, 0
for pair in freqs.keys():
    if pair[1] > 0:
        N_pos += freqs[pair]
    else:
        N_neg += freqs[pair]
loglikelihood = {}
for word in vocab:
    freq_pos = freqs.get((word, 1.0), 0)
    freq_neg = freqs.get((word, 0.0), 0)
    p_w_pos = (freq_pos + 1) / (N_pos + V)
    p_w_neg = (freq_neg + 1) / (N_neg + V)
    loglikelihood[word] = np.log(p_w_pos / p_w_neg)

nb_pred = [1 if pu.naive_bayes_predict(p, log_prior, loglikelihood) > 0 else 0 for p in ps_test]

print(f"Overall accuracy: {accuracy_score(y_test, nb_pred):.4f}")
print(f"F1 score: {f1_score(y_test, nb_pred):.4f}")
print(f"Recall on positive psalms score: {recall_score(y_test, nb_pred):.4f}")
print(f"ROC AUC score: {roc_auc_score(y_test, nb_pred):.4f}")
misclassified = sum(1 for pred, actual in zip(nb_pred, y_test) if pred != actual)
print(f"Misclassified: {misclassified} out of {len(ps_test)} ({misclassified/len(ps_test):.2%})")

Overall accuracy: 0.9447
F1 score: 0.9595
Recall on positive psalms score: 0.9686
ROC AUC score: 0.9316
Misclassified: 13 out of 235 (5.53%)
